# Figure pieces

Renders and schematics for the architecture figure, all written transparent for Inkscape.

PyVista pieces (need a mesh): `10_scene_3d.png`, `11_faceon.png`, `13_triplanar_corner.png`.
Matplotlib schematics (standalone): `cnn_blocks.svg`, `mlp_blocks.svg`, `reshape_2x2.svg`,
`triplanar.svg`, `plane_features.svg`.

Cell 1 holds every import and shared parameter; cell 2 every function. Parameters that are
worth tuning per figure sit with that figure.

## 1. Imports and shared parameters

In [ ]:
from pathlib import Path

import numpy as np
import trimesh
import trimesh.transformations as tf
import pyvista as pv
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.path import Path as MPLPath
from matplotlib.patches import FancyBboxPatch, Polygon

pv.OFF_SCREEN = True
matplotlib.rcParams["svg.fonttype"] = "none"      # editable text in Inkscape

# ----------------------------------------------------------------- input/output
MESH_PATH = "/home/k.wolcott/NSM/nsm/vertebrae_meshes/agamidae_agama_atra_uf180711_01-c3.vtk"
OUT       = Path("svg_pieces")
WINDOW    = (1400, 1400)

# ------------------------------------------------------------- mesh orientation
PRE_ROTATE_DEG  = 90
PRE_ROTATE_AXIS = 2                  # 0=x, 1=y, 2=z

# ---------------------------------------------------------------- cutting plane
SLAB_AXIS = 1                        # plane normal: 0=x, 1=y, 2=z
PLOT_AXES = (0, 2)                   # in-plane axes; must exclude SLAB_AXIS

# SLICE_ABSOLUTE wins when not None; otherwise the fraction is used, where
# 0.0 and 1.0 are the ends of the mesh along SLAB_AXIS.
SLICE_FRACTION = 0.5
SLICE_ABSOLUTE = -0.0157141550628499

# ------------------------------------------------------ point sampling (File S2)
SIGMA_NEAR = 0.015811388300841896
SIGMA_FAR  = 0.05
FRAC_NEAR, FRAC_FAR = 0.40, 0.40     # remainder is uniform in the bounding box
N_POINTS   = 500
SEED       = 52122

# ---------------------------------------------------------------------- palette
C_XY, C_XZ, C_YZ = "#FF9500", "#B056FF", "#25998F"    # the three planes
QUERY_COLOR      = "#FF187C"
EDGE             = "#000000"
BLOCK_COLOR      = "#099816"                          # network blocks
BLOCK_BACK       = "#94D399"

MESH_COLOR    = "#CAA57E"
MESH_OPACITY  = 0.6
INSIDE_COLOR  = "#FF9500"
OUTSIDE_COLOR = "#B056FF"
POINT_SIZE    = 20

# ----------------------------------------------------------------------- camera
CAM_START     = "xz"                 # base view, before azimuth/elevation
CAM_AZIMUTH   = 35
CAM_ELEVATION = 18
CAM_ZOOM      = 1.25

## 2. Functions

In [ ]:
# ---------------------------------------------------------------- mesh handling
def load_mesh(path, weld=True):
    # PyVista, not trimesh: trimesh cannot read POLYDATA-flavoured .vtk.
    # Welding merges coincident vertices, which some exports leave split.
    surf = pv.read(Path(path))
    if not isinstance(surf, pv.PolyData):
        try:
            surf = surf.extract_surface(algorithm="dataset_surface")
        except TypeError:
            surf = surf.extract_surface()
    surf = surf.triangulate()
    if weld:
        surf = surf.clean()
    faces = surf.faces.reshape(-1, 4)[:, 1:]
    return trimesh.Trimesh(np.asarray(surf.points), np.asarray(faces), process=False)


def prepare_mesh(path, rotate_deg=0, rotate_axis=2):
    # Rotate here so bounds, section, points and every render share one orientation.
    m = load_mesh(path)
    m.apply_translation(-m.vertices.mean(axis=0))
    m.apply_scale(1.0 / np.linalg.norm(m.vertices, axis=1).max())
    if rotate_deg:
        m.apply_transform(tf.rotation_matrix(np.radians(rotate_deg),
                                             np.eye(3)[rotate_axis]))
    return m


def to_polydata(m):
    return pv.PolyData(m.vertices,
                       np.hstack([np.full((len(m.faces), 1), 3), m.faces]).ravel())


# --------------------------------------------------------------- section & points
def slice_coord(m, axis, fraction=None, absolute=None):
    if absolute is not None:
        return float(absolute)
    lo, hi = m.bounds[0][axis], m.bounds[1][axis]
    return lo + fraction * (hi - lo)


def section_loops(m, axis, coord, plot_axes):
    normal = np.eye(3)[axis]
    sec = m.section(plane_origin=normal * coord, plane_normal=normal)
    if sec is None:
        raise RuntimeError("empty section - adjust the slice position or SLAB_AXIS")
    return [p[:, list(plot_axes)] for p in sec.discrete if len(p) > 3]


def resample(poly, n):
    d = np.r_[0, np.cumsum(np.linalg.norm(np.diff(poly, axis=0), axis=1))]
    t = np.linspace(0, d[-1], n)
    return np.column_stack([np.interp(t, d, poly[:, i]) for i in (0, 1)])


def sample_points(boundary, n, rng):
    n_near, n_far = int(round(FRAC_NEAR * n)), int(round(FRAC_FAR * n))
    n_unif = n - n_near - n_far
    pick = lambda k: boundary[rng.integers(0, len(boundary), k)]
    return np.vstack([
        pick(n_near) + rng.normal(0, SIGMA_NEAR, (n_near, 2)),
        pick(n_far)  + rng.normal(0, SIGMA_FAR,  (n_far, 2)),
        rng.uniform(boundary.min(0), boundary.max(0), (n_unif, 2)),
    ])


def inside_loops(p2, loops):
    # XOR so an enclosed hole such as the neural canal reads as outside. Tested
    # against the drawn curve, so no watertight mesh is required.
    c = np.zeros(len(p2), bool)
    for poly in loops:
        c ^= MPLPath(poly).contains_points(p2)
    return c


def lift(p2, plot_axes, axis, coord):
    p3 = np.empty((len(p2), 3))
    p3[:, plot_axes[0]] = p2[:, 0]
    p3[:, plot_axes[1]] = p2[:, 1]
    p3[:, axis] = coord
    return p3


# ------------------------------------------------------------------- pyvista draw
def add_points(pl, pts, mask, size=POINT_SIZE, lighting=True,
               inside_color=None, outside_color=None):
    inside_color  = inside_color  or INSIDE_COLOR
    outside_color = outside_color or OUTSIDE_COLOR
    for k, color in ((mask, inside_color), (~mask, outside_color)):
        if k.any():
            pl.add_mesh(pv.PolyData(pts[k]), color=color, point_size=size,
                        render_points_as_spheres=True, lighting=lighting)


def quad(axis, corner, sign, size):
    # Square in the plane, spanning from `corner` into the chosen octant.
    others = [i for i in range(3) if i != axis]
    pts = np.zeros((4, 3)); pts[:] = corner
    for k, (u, v) in enumerate([(0, 0), (1, 0), (1, 1), (0, 1)]):
        pts[k, others[0]] = corner[others[0]] + sign[others[0]] * u * size
        pts[k, others[1]] = corner[others[1]] + sign[others[1]] * v * size
    return pv.PolyData(pts, np.array([4, 0, 1, 2, 3]))


def gt_points(axis, corner, sign, size, n, rng, margin=0.06):
    others = [i for i in range(3) if i != axis]
    p = np.zeros((n, 3)); p[:, axis] = corner[axis]
    for o in others:
        p[:, o] = corner[o] + sign[o] * rng.uniform(margin, 1 - margin, n) * size
    return p


def show(path, title):
    fig, a = plt.subplots(figsize=(5, 5))
    a.imshow(plt.imread(path)); a.set_title(title, fontsize=9); a.axis("off")
    plt.show()


# ---------------------------------------------------------------- matplotlib draw
def block_row(ax, heights, width, gap, rounding, lw, color, edge=EDGE):
    # Row of rounded rectangles, centred vertically.
    for i, h in enumerate(heights):
        ax.add_patch(FancyBboxPatch(
            (i * (width + gap), -h / 2), width, h,
            boxstyle=f"round,pad=0,rounding_size={rounding}",
            facecolor=color, edgecolor=edge, linewidth=lw))


def grid_2x2(ax, off, cell, cgap, rounding, face, lw, z, edge=EDGE):
    for r in range(2):
        for c in range(2):
            ax.add_patch(FancyBboxPatch(
                (c * (cell + cgap) + off, r * (cell + cgap) + off), cell, cell,
                boxstyle=f"round,pad=0,rounding_size={rounding}",
                facecolor=face, edgecolor=edge, linewidth=lw, zorder=z))


def column(ax, x, color, n, cw, ch, cgap, rounding, lw, label=None, edge=EDGE):
    for i in range(n):
        ax.add_patch(FancyBboxPatch((x, i * (ch + cgap)), cw, ch,
            boxstyle=f"round,pad=0,rounding_size={rounding}",
            facecolor=color, edgecolor=edge, linewidth=lw, zorder=3))
    if label:
        ax.text(x + cw / 2, -0.16, label, ha="center", va="top", fontsize=10)
    return x + cw


def make_projector(rot_deg, rot_axis, ex, ey, ez):
    # Returns P(v): rotate then project 3D -> 2D, always shape (n, 2).
    t = np.radians(rot_deg); c, s = np.cos(t), np.sin(t)
    R = np.eye(3)
    a, b = [i for i in range(3) if i != rot_axis]
    R[a, a] = c; R[a, b] = -s
    R[b, a] = s; R[b, b] = c

    def P(v):
        v = np.atleast_2d(np.asarray(v, float)) @ R.T
        return v[:, 0:1]*ex + v[:, 1:2]*ey + v[:, 2:3]*ez
    return P

## 3. Build the geometry

In [ ]:
OUT.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)

mesh = prepare_mesh(MESH_PATH, PRE_ROTATE_DEG, PRE_ROTATE_AXIS)

SECTION_ORIGIN = slice_coord(mesh, SLAB_AXIS, SLICE_FRACTION, SLICE_ABSOLUTE)
normal = np.eye(3)[SLAB_AXIS]
origin = normal * SECTION_ORIGIN

loops    = section_loops(mesh, SLAB_AXIS, SECTION_ORIGIN, PLOT_AXES)
boundary = np.vstack([resample(p, 600) for p in loops])

pts2  = sample_points(boundary, N_POINTS, rng)
is_in = inside_loops(pts2, loops)
pts3  = lift(pts2, PLOT_AXES, SLAB_AXIS, SECTION_ORIGIN)

surf       = to_polydata(mesh)
slice_line = surf.slice(normal=normal, origin=origin)

print(f"faces {len(mesh.faces):,} | {len(loops)} loop(s) | "
      f"cut at {'xyz'[SLAB_AXIS]}={SECTION_ORIGIN:.4f}")
print(f"{len(pts3)} points | {100*is_in.mean():.0f}% inside")

## 4. Piece 10 — angled scene

In [ ]:
PLANE_COLOR      = "#B0B0B0"
PLANE_OPACITY    = 0.0
PLANE_SPAN       = 1.2               # multiple of the mesh extent
PLANE_EDGE_COLOR = "#6E6E6E"
PLANE_EDGE_WIDTH = 10

# i_resolution/j_resolution default to 10, which would draw a 10x10 grid of edges.
span  = float(np.ptp(mesh.bounds, axis=0).max()) * PLANE_SPAN
plane = pv.Plane(center=origin, direction=normal, i_size=span, j_size=span,
                 i_resolution=1, j_resolution=1)
plane_edge = plane.extract_feature_edges(boundary_edges=True, feature_edges=False,
                                         manifold_edges=False, non_manifold_edges=False)

pl = pv.Plotter(off_screen=True, window_size=WINDOW)
pl.set_background("white")
pl.add_mesh(surf, color=MESH_COLOR, opacity=MESH_OPACITY,
            smooth_shading=True, specular=0.25)
#pl.add_mesh(plane, color=PLANE_COLOR, opacity=PLANE_OPACITY, lighting=False)
pl.add_mesh(plane_edge, color=PLANE_EDGE_COLOR, line_width=PLANE_EDGE_WIDTH,
            lighting=False)
add_points(pl, pts3, is_in)

pl.camera_position  = CAM_START
pl.camera.azimuth   = CAM_AZIMUTH
pl.camera.elevation = CAM_ELEVATION
pl.camera.zoom(CAM_ZOOM)

p10 = OUT / "10_scene_3d.png"
pl.screenshot(str(p10), transparent_background=True); pl.close()
print(p10); show(p10, "10  angled scene")

## 5. Piece 11 — face-on slice

In [ ]:
FACE_INSIDE_COLOR  = "#c95700"
FACE_OUTSIDE_COLOR = "#4B0C82"
POINT_SIZE_FACE    = 30
OUTLINE_COLOR      = "#332c25"
OUTLINE_WIDTH      = 20
FACE_ZOOM          = 1.3

pl = pv.Plotter(off_screen=True, window_size=WINDOW)
pl.set_background("white")
pl.add_mesh(slice_line, color=OUTLINE_COLOR, line_width=OUTLINE_WIDTH)
add_points(pl, pts3, is_in, lighting=False, size=POINT_SIZE_FACE,
           inside_color=FACE_INSIDE_COLOR, outside_color=FACE_OUTSIDE_COLOR)

# Parallel projection: no perspective, so the outline keeps its true shape.
pl.enable_parallel_projection()
dist = float(np.ptp(mesh.bounds, axis=0).max()) * 3
pl.camera_position = [tuple(origin + normal * dist), tuple(origin),
                      tuple(np.eye(3)[PLOT_AXES[1]])]
pl.reset_camera(); pl.camera.zoom(FACE_ZOOM)

p11 = OUT / "11_faceon.png"
pl.screenshot(str(p11), transparent_background=True); pl.close()
print(p11); show(p11, "11  face-on slice")

## 6. Piece 13 — corner planes with real traces

In [ ]:
PLANE_COLORS = {0: C_YZ, 1: C_XZ, 2: C_XY}      # key = the plane's NORMAL axis
CORNER       = np.array([0.1, -0.2, -0.3])      # where the three quads meet
SIGN         = np.array([1.0, 1.0, 1.0])        # which octant they occupy (+/-1)
SIZE_MULT    = 0.55                             # quad edge, fraction of mesh extent
PLANE_ALPHA  = 0.5
TRACE_WIDTH  = 15
CLIP_PAD     = 0.02

MESH_ROT_DEG  = 90                              # this piece only
MESH_ROT_AXIS = 2

N_GT       = 45
GT_COLOR   = "#8A8A8A"
GT_SIZE    = 20
GT_ALPHA   = 0.45
GT_SEED    = 3
QUERY_FRAC = np.array([0.45, 0.55, 0.40])       # query position within the octant
QUERY_SIZE = 40

AXIS_LEN     = 1.3                              # multiple of SIZE
AXIS_COLOR   = "#000000"
AXIS_SHAFT_R = 0.010
AXIS_TIP_R   = 0.045
AXIS_TIP_L   = 0.13
AXIS_LABELS  = ("x", "y", "z")
AXIS_FONT    = 34
LABEL_OFFSET = 1.10
VIEW_AZ      = CAM_AZIMUTH + 100

SIZE   = float(np.ptp(mesh.bounds, axis=0).max()) * SIZE_MULT
QUERY  = CORNER + SIGN * SIZE * QUERY_FRAC
rng_gt = np.random.default_rng(GT_SEED)

rotate = {0: "rotate_x", 1: "rotate_y", 2: "rotate_z"}[MESH_ROT_AXIS]
surf_r = (getattr(surf, rotate)(MESH_ROT_DEG, point=(0, 0, 0), inplace=False)
          if MESH_ROT_DEG else surf)

lo = np.minimum(CORNER, CORNER + SIGN*SIZE) - CLIP_PAD
hi = np.maximum(CORNER, CORNER + SIGN*SIZE) + CLIP_PAD
clip_box = [lo[0], hi[0], lo[1], hi[1], lo[2], hi[2]]

pl = pv.Plotter(off_screen=True, window_size=WINDOW)
pl.set_background("white")

for axis, color in PLANE_COLORS.items():
    pl.add_mesh(quad(axis, CORNER, SIGN, SIZE),
                color=color, opacity=PLANE_ALPHA, lighting=False)

    sec = surf_r.slice(normal=np.eye(3)[axis], origin=tuple(CORNER))
    if sec.n_points:
        clipped = sec.clip_box(clip_box, invert=False)   # keep the part on the quad
        if clipped.n_points:
            pl.add_mesh(clipped, color=color, line_width=TRACE_WIDTH, lighting=False)
    else:
        print(f"  axis {axis}: empty section")

    pl.add_mesh(pv.PolyData(gt_points(axis, CORNER, SIGN, SIZE, N_GT, rng_gt)),
                color=GT_COLOR, point_size=GT_SIZE, opacity=GT_ALPHA,
                render_points_as_spheres=True, lighting=False)

    q = QUERY.copy(); q[axis] = CORNER[axis]             # project onto this plane
    pl.add_mesh(pv.PolyData(q.reshape(1, 3)), color=QUERY_COLOR,
                point_size=QUERY_SIZE, render_points_as_spheres=True,
                lighting=False)

tips = []
for axis, lab in enumerate(AXIS_LABELS):
    d = np.eye(3)[axis] * SIGN[axis]
    L = SIZE * AXIS_LEN
    pl.add_mesh(pv.Arrow(start=tuple(CORNER), direction=tuple(d), scale=L,
                         tip_length=AXIS_TIP_L, tip_radius=AXIS_TIP_R,
                         shaft_radius=AXIS_SHAFT_R),
                color=AXIS_COLOR, lighting=False)
    tips.append(CORNER + d * L * LABEL_OFFSET)

pl.add_point_labels(np.array(tips), list(AXIS_LABELS), font_size=AXIS_FONT,
                    text_color=AXIS_COLOR, shape=None, show_points=False,
                    always_visible=True)

pl.camera_position  = CAM_START
pl.camera.azimuth   = VIEW_AZ
pl.camera.elevation = CAM_ELEVATION
pl.camera.zoom(CAM_ZOOM)

p13 = OUT / "13_triplanar_corner.png"
pl.screenshot(str(p13), transparent_background=True); pl.close()
print(p13); show(p13, "13  corner planes with anatomical traces")

## 7. Schematics

### CNN blocks

In [ ]:
N     = 5
H_MIN, H_MAX = 0.6, 2.0
WIDTH = 0.22
GAP   = 0.22
ROUND = 0.06
LW    = 3.0

fig, ax = plt.subplots(figsize=(6, 3))
block_row(ax, np.linspace(H_MIN, H_MAX, N), WIDTH, GAP, ROUND, LW, BLOCK_COLOR)
ax.set_xlim(-0.2, N * (WIDTH + GAP))
ax.set_ylim(-H_MAX/2 - 0.3, H_MAX/2 + 0.3)
ax.set_aspect("equal"); ax.axis("off")
fig.savefig(OUT/"cnn_blocks.svg", transparent=True, bbox_inches="tight")
plt.show()

### Dense

In [ ]:
N      = 1
HEIGHT = 1.4       # same for every layer: sdf_hidden_dims = [512, 512, 512]
WIDTH  = 0.22
GAP    = 0.22
ROUND  = 0.06
LW     = 3.0

fig, ax = plt.subplots(figsize=(6, 3))
block_row(ax, [HEIGHT] * N, WIDTH, GAP, ROUND, LW, BLOCK_COLOR)
ax.set_xlim(-0.2, N * (WIDTH + GAP))
ax.set_ylim(-HEIGHT/2 - 0.3, HEIGHT/2 + 0.3)
ax.set_aspect("equal"); ax.axis("off")
fig.savefig(OUT/"dense_blocks.svg", transparent=True, bbox_inches="tight")
plt.show()

### MLP

In [ ]:
N      = 3
HEIGHT = 1.4       # same for every layer: sdf_hidden_dims = [512, 512, 512]
WIDTH  = 0.22
GAP    = 0.22
ROUND  = 0.06
LW     = 3.0

fig, ax = plt.subplots(figsize=(6, 3))
block_row(ax, [HEIGHT] * N, WIDTH, GAP, ROUND, LW, BLOCK_COLOR)
ax.set_xlim(-0.2, N * (WIDTH + GAP))
ax.set_ylim(-HEIGHT/2 - 0.3, HEIGHT/2 + 0.3)
ax.set_aspect("equal"); ax.axis("off")
fig.savefig(OUT/"mlp_blocks.svg", transparent=True, bbox_inches="tight")
plt.show()

### Reshape 2×2

In [ ]:
CELL  = 0.30
CGAP  = 0.05
ROUND = 0.05
LW    = 3.0
DEPTH = 0.06       # offset per back copy
NBACK = 2

fig, ax = plt.subplots(figsize=(3, 3))
for k in range(NBACK, 0, -1):                                    # back to front
    grid_2x2(ax, k * DEPTH, CELL, CGAP, ROUND, BLOCK_BACK, LW * 0.7, -k)
grid_2x2(ax, 0.0, CELL, CGAP, ROUND, BLOCK_COLOR, LW, 1)         # front face

span_2x2 = 2 * CELL + CGAP
ax.set_xlim(-0.1, span_2x2 + NBACK * DEPTH + 0.1)
ax.set_ylim(-0.1, span_2x2 + NBACK * DEPTH + 0.1)
ax.set_aspect("equal"); ax.axis("off")
fig.savefig(OUT/"reshape_2x2.svg", transparent=True, bbox_inches="tight")
plt.show()

### Triplanar

In [ ]:
EX = np.array([ 1.00, -0.30])       # axonometric projection: 3D -> 2D
EY = np.array([ 0.55,  0.42])
EZ = np.array([ 0.00,  1.00])
ROT_DEG  = 260                      # rotate the whole scene
ROT_AXIS = 2
S        = 1.0                      # plane size
PT       = np.array([0.52, 0.52, 0.52])
ALPHA    = 0.55
AXIS_LEN = 1.3

P = make_projector(ROT_DEG, ROT_AXIS, EX, EY, EZ)

# corners, colour, and where the query point lands on that plane
quads = {
    "xy": ([[0,0,0],[S,0,0],[S,S,0],[0,S,0]], C_XY, [PT[0], PT[1], 0]),
    "xz": ([[0,0,0],[S,0,0],[S,0,S],[0,0,S]], C_XZ, [PT[0], 0, PT[2]]),
    "yz": ([[0,0,0],[0,S,0],[0,S,S],[0,0,S]], C_YZ, [0, PT[1], PT[2]]),
}

fig, ax = plt.subplots(figsize=(5, 5))

for corners, color, _ in quads.values():
    ax.add_patch(Polygon(P(np.array(corners)), closed=True, facecolor=color,
                         edgecolor=EDGE, alpha=ALPHA, lw=0, zorder=1))

for vec, lab in ((np.array([S*AXIS_LEN, 0, 0]), "x"),
                 (np.array([0, S*AXIS_LEN, 0]), "z"),
                 (np.array([0, 0, S*AXIS_LEN]), "y")):
    a, b = P(np.zeros(3))[0], P(vec)[0]
    ann = ax.annotate("", xy=b, xytext=a,
                      arrowprops=dict(arrowstyle="-|>,head_width=0.5,head_length=0.9",
                                      lw=5, color=EDGE, shrinkA=0, shrinkB=0),
                      annotation_clip=False, zorder=4)
    ann.arrow_patch.set_clip_on(False)          # arrows extend past the planes
    d = b - a; n = np.linalg.norm(d)
    ax.text(*(b + 0.12*d/max(n, 1e-9)), lab, fontsize=11, ha="center", va="center")

for corners, color, proj in quads.values():
    q = P(np.array(proj))[0]
    ax.plot(*q, "o", ms=15, color=QUERY_COLOR, mec=EDGE, mew=3, zorder=5)

ax.set_aspect("equal"); ax.axis("off"); ax.autoscale_view()
fig.savefig(OUT/"triplanar.svg", transparent=True, bbox_inches="tight")
plt.show()

### Plane features

In [ ]:
SUM_COLOR = QUERY_COLOR
NCELL     = 4        # cells drawn per vector (schematic, not the real 128)
CW, CH    = 0.42, 0.26
CGAP      = 0.03
COLGAP    = 0.34
ROUND     = 0.04
LW        = 3.0
SHOW_SUM  = True

fig, ax = plt.subplots(figsize=(5, 3))

x = 0.0
for color, lab in ((C_XY, "$xy$"), (C_XZ, "$xz$"), (C_YZ, "$yz$")):
    x = column(ax, x, color, NCELL, CW, CH, CGAP, ROUND, LW, label=lab) + COLGAP

top = NCELL*(CH + CGAP) - CGAP
if SHOW_SUM:
    sx = x - COLGAP + 0.10
    ax.text(sx + 0.18, top/2, r"$\Sigma$", fontsize=22, ha="center", va="center")
    ax.annotate("", xy=(sx + 0.55, top/2), xytext=(sx + 0.36, top/2),
                arrowprops=dict(arrowstyle="-|>", lw=1.6, color="#666666"))
    column(ax, sx + 0.62, SUM_COLOR, NCELL, CW, CH, CGAP, ROUND, LW,
           label="local $z$")
    x = sx + 0.62 + CW

ax.set_xlim(-0.15, x + 0.15)
ax.set_ylim(-0.75, top + 0.15)
ax.set_aspect("equal"); ax.axis("off")
fig.savefig(OUT/"plane_features.svg", transparent=True, bbox_inches="tight")
plt.show()

### Global Z

In [ ]:
SUM_COLOR = QUERY_COLOR
NCELL     = 8        # cells drawn per vector (schematic, not the real 128)
CW, CH    = 0.42, 0.26
CGAP      = 0.03
COLGAP    = 0.34
ROUND     = 0.04
LW        = 3.0
SHOW_SUM  = True

fig, ax = plt.subplots(figsize=(5, 3))

x = 0.0
lab = "global z"
color = "#F6e5cc"
x = column(ax, x, color, NCELL, CW, CH, CGAP, ROUND, LW, label=lab) + COLGAP


ax.set_xlim(-0.15, x + 0.15)
ax.set_ylim(-0.75, top + 0.15)
ax.set_aspect("equal"); ax.axis("off")
fig.savefig(OUT/"global_z.svg", transparent=True, bbox_inches="tight")
plt.show()